# SORA · Pelatihan Model Segmentasi Sampah (DeepLabv3+)

Notebook ini melatih model yang dipakai aplikasi petugas SORA untuk mengenali enam fraksi sampah dari kamera HP:
organik basah, plastik, kertas dan kardus, tekstil, kayu dan ranting, serta inert (ditambah kelas latar).

**Alur:** foto sampah + anotasi LabelMe → masker → pembagian data per muatan yang dibekukan (StratifiedGroupKFold per lokasi + set uji akhir) → pelatihan DeepLabv3+ ResNet-50 →
evaluasi (IoU, mIoU, MAE proporsi, matriks kebingungan) → ekspor ONNX → salin ke `public/models/` aplikasi.

Semua langkah mengikuti lampiran laporan Bab 3 (Subbab 3.2–3.4 dan 3.9).

**Sebelum mulai:** Runtime → Change runtime type → **T4 GPU**.

| Data | Isi | Target tahap 1 |
|---|---|---|
| D1 | Foto sampah + poligon LabelMe | ≥ 300 foto dari ≥ 60 muatan/porsi |
| D3 | Uji oven (untuk kalibrasi kadar air, bagian akhir) | ≥ 30 muatan |

Konvensi nama foto: `<ID_MUATAN>_<TPS>_f<NN>.jpg`, misalnya `M001_PasarKudus_f01.jpg`. Semua foto dari satu muatan memakai ID yang sama.

## 1. Siapkan kode dan pustaka

In [ ]:
# Pilihan A: repo aplikasi sudah memuat folder training/
REPO = 'https://github.com/azunzios/karta-mobile'
!git clone -q $REPO /content/karta-mobile || echo 'Repo sudah ada'
%cd /content/karta-mobile/training
# Pilihan B: bila folder training/ belum ada di repo, unggah training.zip lalu:
# !unzip -q /content/training.zip -d /content && %cd /content/training
!ls

In [ ]:
!pip -q install segmentation-models-pytorch==0.3.4 onnx onnxruntime opencv-python-headless scikit-learn matplotlib
import torch; print('GPU:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

## 2. Hubungkan Google Drive dan ubah anotasi LabelMe menjadi masker
Simpan foto `.jpg` dan anotasi `.json` LabelMe dalam satu folder di Drive, misalnya `MyDrive/SORA/anotasi`.
Nama label yang dikenali ada di `karta_kelas.py` (organik, plastik, kertas, kardus, tekstil, kayu, ranting, inert, batu, kaca, logam, abaikan, dan lain-lain).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
ANOTASI = '/content/drive/MyDrive/SORA/anotasi'   # ubah sesuai folder tim
DATA = '/content/data/siap'
!python labelme_ke_mask.py --masuk "$ANOTASI" --keluar $DATA

### Periksa data: jumlah piksel per kelas dan contoh masker

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from PIL import Image
from karta_kelas import NAMA, WARNA
meta = pd.read_csv(f'{DATA}/meta.csv')
print(len(meta), 'foto dari', meta.id_muatan.nunique(), 'muatan;', meta.tps.nunique(), 'TPS')
hit = np.zeros(7)
for f in meta.foto:
    m = np.array(Image.open(f'{DATA}/masker/{f}.png')); hit += np.bincount(m[m < 7].ravel(), minlength=7)
display(pd.DataFrame({'kelas': NAMA, 'persen_piksel': (hit / hit.sum() * 100).round(1)}))
pal = np.array(WARNA, np.uint8)
fig, ax = plt.subplots(2, 3, figsize=(12, 7))
for j, f in enumerate(meta.foto.sample(min(3, len(meta)), random_state=1)):
    ax[0, j].imshow(Image.open(f'{DATA}/gambar/{f}.jpg')); ax[0, j].set_title(f, fontsize=8)
    m = np.array(Image.open(f'{DATA}/masker/{f}.png')); ax[1, j].imshow(pal[np.minimum(m, 6)])
[a.axis('off') for a in ax.ravel()]; plt.show()

## 3. (Opsional) Pralatih dengan dataset publik ZeroWaste-f dan TACO
Lampiran Subbab 3.1.6. Keduanya **hanya** untuk pralatih, tidak pernah untuk validasi atau uji.
- **ZeroWaste-f**: 1.245 frame latih konveyor daur ulang. Lisensi CC BY-NC 4.0 (nonkomersial): model yang dijual nanti harus dilatih ulang tanpa ZeroWaste.
- **TACO**: 1.500 foto sampah berserakan, 60 kategori.
Manfaatnya diuji lewat eksperimen A2 (bandingkan hasil dengan dan tanpa pralatih).

In [ ]:
PAKAI_ZEROWASTE = False   # unduh zerowaste-f-final dari situs ZeroWaste, taruh di Drive
PAKAI_TACO = False
ZW = '/content/drive/MyDrive/SORA/zerowaste-f-final'
sumber = []
if PAKAI_ZEROWASTE:
    !python zerowaste_ke_karta.py --anotasi "$ZW/train/labels.json" --gambar "$ZW/train/data" --keluar /content/data/zerowaste
    sumber.append('/content/data/zerowaste')
if PAKAI_TACO:
    !git clone -q https://github.com/pedropro/TACO /content/TACO
    !cd /content/TACO && pip -q install -r requirements.txt && python download.py
    !python taco_ke_karta.py --anotasi /content/TACO/data/annotations.json --gambar /content/TACO/data --keluar /content/data/taco
    sumber.append('/content/data/taco')
# gabungkan menjadi satu folder pralatih
import shutil, os
if sumber:
    G = '/content/data/pralatih'
    for sub in ('gambar', 'masker'): os.makedirs(f'{G}/{sub}', exist_ok=True)
    semua = []
    for s in sumber:
        m = pd.read_csv(f'{s}/meta.csv')
        for f in m.foto:
            shutil.copy(f'{s}/gambar/{f}.jpg', f'{G}/gambar/'); shutil.copy(f'{s}/masker/{f}.png', f'{G}/masker/')
        semua.append(m)
    pd.concat(semua).to_csv(f'{G}/meta.csv', index=False)
    print(len(pd.concat(semua)), 'foto pralatih')

## 4. (Opsional) Uji alur dengan data sintetis
Bila foto tim belum siap, sel ini membuat data tekstur buatan untuk memastikan seluruh alur berjalan. Model dari data ini **bukan** model pengenal sampah dan tidak boleh dilaporkan sebagai hasil.

In [ ]:
UJI_ALUR = False
if UJI_ALUR:
    !python buat_data_sintetis.py --keluar /content/data/sintetis --muatan 64 --frame 5 --ukuran 256
    DATA = '/content/data/sintetis'

## 5. Bekukan pembagian data (sekali saja)
Lampiran Subbab 3.3: set uji akhir 3 unit per lokasi (6 untuk campuran buatan, kolom `tps` = `Campuran`), dipilih dari unit yang diuji oven bila kolom `oven` ada; sisanya 5 lipatan StratifiedGroupKFold per lokasi.
Sel ini menolak berjalan bila `meta.csv` sudah punya kolom `set`, agar pembagian tidak berubah-ubah. **Catat SHA-256 yang dicetak ke laporan.**

In [ ]:
!python buat_split.py --data $DATA --uji-per-lokasi 3 --khusus Campuran=6 --lipatan 5
!cp $DATA/split_v1.csv /content/drive/MyDrive/SORA/ 2>/dev/null || true

## 6. Latih dan evaluasi
- `--cv 5`: validasi silang 5 lipatan (lipatan dari langkah 5), rata-rata ± simpangan baku.
- `--lolo`: validasi tinggalkan-satu-lokasi (campuran buatan selalu ikut dilatih).
- Model akhir dilatih ulang pada **seluruh** data pengembangan selama median epoch terbaik validasi silang, lalu set uji akhir dibuka **satu kali** dengan selang kepercayaan 95% bootstrap per muatan.
- `--kuantisasi`: membuat versi INT8 (±4× lebih kecil) untuk HP.

In [ ]:
PRALATIH = '--pralatih /content/data/pralatih --epoch-pralatih 20' if sumber else ''
LABEL = 'sintetis-uji-alur' if UJI_ALUR else 'foto-tim'
!python latih.py --data $DATA --keluar /content/hasil --encoder resnet50 --bobot imagenet \
    --ukuran 512 --epoch 80 --batch 8 --cv 5 --lolo --kuantisasi --label-data $LABEL $PRALATIH

## 7. Lihat hasil evaluasi

In [ ]:
import json
ev = json.load(open('/content/hasil/evaluasi.json'))
u = ev['uji_akhir']
print(f"Set uji: {u['jumlah_frame']} frame dari {u['jumlah_muatan']} muatan")
print(f"mIoU = {u['miou']:.3f} (target >= 0,55)   MAE proporsi = {u['mae_proporsi_pp']:.2f} poin persen (target <= 3)")
if 'bootstrap' in u:
    b = u['bootstrap']; print(f"  CI95 bootstrap per muatan: mIoU {b['miou_ci95']}, MAE proporsi {b['mae_proporsi_ci95_pp']} pp")
print('SHA-256 pembagian:', ev['data'].get('sha256_pembagian'))
display(pd.DataFrame({'IoU': u['iou_per_kelas']}).T)
display(pd.DataFrame({'MAE proporsi (pp)': u['mae_proporsi_per_fraksi_pp']}).T)
if 'validasi_silang' in ev:
    v = ev['validasi_silang']
    print(f"Validasi silang: mIoU {v['miou_rerata']:.3f} ± {v['miou_sb']:.3f}; MAE proporsi {v['mae_proporsi_rerata_pp']:.2f} ± {v['mae_proporsi_sb_pp']:.2f} pp")
if 'tinggalkan_satu_lokasi' in ev:
    display(pd.DataFrame({l: {'mIoU': x['miou'], 'MAE (pp)': x['mae_proporsi_pp'], 'muatan': x['muatan']} for l, x in ev['tinggalkan_satu_lokasi']['per_lokasi'].items()}).T)
k = pd.DataFrame(ev['kurva'])
if len(k): k.plot(x='epoch', y=[c for c in ('loss', 'miou_val') if c in k], figsize=(7, 3)); plt.show()
display(Image.open('/content/hasil/matriks_kebingungan.png'))
print(ev['ekspor'])

## 8. Pasang model ke aplikasi petugas
Salin dua berkas ke folder `public/models/` di repo aplikasi, lalu deploy ulang (Vercel):
- `karta-seg.json`
- berkas ONNX yang disebut pada kolom `berkas` di JSON (`karta-seg-int8.onnx` bila kuantisasi lolos, selain itu `karta-seg.onnx`)

In [ ]:
import shutil, os
TUJUAN = '/content/drive/MyDrive/SORA/model_aplikasi'; os.makedirs(TUJUAN, exist_ok=True)
meta_m = json.load(open('/content/hasil/karta-seg.json'))
for f in ['karta-seg.json', meta_m['berkas'], 'evaluasi.json', 'matriks_kebingungan.png']:
    shutil.copy(f'/content/hasil/{f}', TUJUAN)
print('Tersalin ke', TUJUAN, '->', os.listdir(TUJUAN))

## 9. Kalibrasi model kadar air dari uji oven
Isi `templat/uji_oven.csv` (satu baris per muatan yang diuji oven, metode ASTM E790-15 pada 110 °C). Komposisi diisi persen massa hasil pilah-timbang. Hasilnya `kadar-air.json`, salin juga ke `public/models/`.

In [ ]:
UJI_OVEN = '/content/drive/MyDrive/SORA/uji_oven.csv'
!python kalibrasi_kadar_air.py --data "$UJI_OVEN" --keluar /content/hasil/kadar-air.json
# Tanpa data, coba dulu dengan data rekaan:  !python kalibrasi_kadar_air.py --demo 40 --keluar /content/hasil/demo.json